# Cu FEFF 64D weighted-shell encoder replication sweep

Train two independent Cu FEFF encoders (seeds 43 and 44). Each encoder exports 192D absorbing-site and weighted-shell features, then trains ten ExpertXAS heads spanning dropout 0.10–0.45 and LR 5e-4–1e-3. The sweep keeps the two best validation-selected settings from the preceding run (dropout 0.20/LR 7e-4 and dropout 0.30/LR 7e-4) while adding nearby broader regularization and LR choices. Compare both 10-head ensembles, the combined 20-head ensemble, and the best validation head using validation eta only. Change `RUN_NAME` for a fresh experiment; reuse it to resume.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import dgl
import lightning.pytorch as pl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from matgl.config import DEFAULT_ELEMENTS
from matgl.ext.pymatgen import Structure2Graph
from matgl.graph.compute import compute_pair_vector_and_distance, compute_theta_and_phi, create_line_graph
from matgl.layers import (
    MLP as M3GNetMLP,
    ActivationFunction,
    BondExpansion,
    EmbeddingBlock,
    GatedMLP,
    M3GNetBlock,
    SphericalBesselWithHarmonics,
    ThreeBodyInteractions,
)
from matgl.utils.cutoff import polynomial_cutoff
from pymatgen.core import Structure
from torch import nn
from torch.utils.data import DataLoader, Dataset

import matgl.layers._basis as matgl_basis
import matgl.layers._three_body as matgl_three_body
import matgl.utils.maths as matgl_math

from omnixas.data.ml_data import MLData, MLSplits
from omnixas.model.xasblock import XASBlock
from omnixas.model.xasblock_regressor import XASBlockRegressor

REPO_ROOT = Path.cwd().resolve()
while not ((REPO_ROOT / "pyproject.toml").exists() and (REPO_ROOT / "omnixas").is_dir()):
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Run this notebook from inside the OmniXAS repository.")
    REPO_ROOT = REPO_ROOT.parent

TASK = "Cu_FEFF"
RUN_NAME = "run_003_two_encoders"
SPLITS = ("train", "val", "test")
FEATURE_SCALE = 1000.0
BATCH_SIZE = 32
NUM_WORKERS = 4
CUTOFF = 5.0
N_BLOCKS = 3
GNN_DROPOUT = 0.1

ENCODER_WIDTH = 64
ENCODER_SEEDS = (43, 44)
ENCODER_EPOCHS = 150
ENCODER_LR = 1e-3
FEATURE_KEY = "site_shells_3_5_weighted"
FEATURE_NAME = f"learned_{FEATURE_KEY}_{ENCODER_WIDTH}d"
FEATURE_DIM = 3 * ENCODER_WIDTH

HEAD_DIMS = [600, 600, 400]
HEAD_EPOCHS = 400
HEAD_CONFIGS = (
    {"seed": 53, "dropout": 0.10, "lr": 7e-4},
    {"seed": 54, "dropout": 0.15, "lr": 1e-3},
    {"seed": 55, "dropout": 0.20, "lr": 5e-4},
    {"seed": 56, "dropout": 0.20, "lr": 7e-4},
    {"seed": 57, "dropout": 0.25, "lr": 1e-3},
    {"seed": 58, "dropout": 0.30, "lr": 5e-4},
    {"seed": 59, "dropout": 0.30, "lr": 7e-4},
    {"seed": 60, "dropout": 0.35, "lr": 1e-3},
    {"seed": 61, "dropout": 0.40, "lr": 7e-4},
    {"seed": 62, "dropout": 0.45, "lr": 1e-3},
)
if len(ENCODER_SEEDS) != 2 or len(set(ENCODER_SEEDS)) != 2:
    raise ValueError("Expected exactly two unique encoder seeds")
if len(HEAD_CONFIGS) != 10 or len({tuple(config.items()) for config in HEAD_CONFIGS}) != 10:
    raise ValueError("Expected exactly 10 unique ExpertXAS configurations")

PAPER_EXPERT_ETA = 5.19
OLD_V1_VAL_ETA = 7.48
OLD_V1_TEST_ETA = 7.64

DATA_DIR = REPO_ROOT / "tutorial_omnixas" / "ml_data"
ID_DIR = REPO_ROOT / "tutorial_omnixas" / "material_id_and_site"
RAW_ROOT = (
    Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data"))
    / "materialscloud_omnixas_raw"
    / "extracted"
)
OUT_ROOT = REPO_ROOT / "output" / "training" / "cuFeffShellEncoderSweep" / RUN_NAME
OUT_ROOT.mkdir(parents=True, exist_ok=True)
if not RAW_ROOT.is_dir():
    raise FileNotFoundError(f"Missing raw OmniXAS data: {RAW_ROOT}")

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("medium")

split_ids = {
    split: [line for line in (ID_DIR / f"{TASK}_{split}.txt").read_text().splitlines() if line]
    for split in SPLITS
}
y_true = {
    split: np.atleast_2d(np.loadtxt(DATA_DIR / f"{TASK}_{split}_y.txt", dtype=np.float32))
    for split in SPLITS
}

for split in SPLITS:
    if len(set(split_ids[split])) != len(split_ids[split]):
        raise ValueError(f"{split}: duplicate material/site IDs")
    expected_shape = (len(split_ids[split]), 141)
    if y_true[split].shape != expected_shape:
        raise ValueError(f"{split}: expected targets with shape {expected_shape}, got {y_true[split].shape}")
    if not np.isfinite(y_true[split]).all():
        raise ValueError(f"{split}: targets contain non-finite values")

materials = {split: {row.rsplit("_", 1)[0] for row in split_ids[split]} for split in SPLITS}
for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    overlap = materials[left] & materials[right]
    if overlap:
        raise ValueError(f"Material leakage between {left} and {right}: {sorted(overlap)[:5]}")

missing_poscars = sorted(
    material_id
    for material_id in set().union(*materials.values())
    if not (RAW_ROOT / "FEFF" / "Cu" / material_id / "POSCAR").is_file()
)
if missing_poscars:
    raise FileNotFoundError(f"Missing Cu FEFF POSCAR files for {missing_poscars[:5]}")


def head_run_name(config):
    return f"seed{config['seed']}_dropout{config['dropout']:g}_lr{config['lr']:g}"


print("repo:", REPO_ROOT)
print("device:", DEVICE)
print("rows:", {split: len(rows) for split, rows in split_ids.items()})


In [ ]:
TRAIN_MEAN = y_true["train"].mean(axis=0, keepdims=True)


def eta_score(prediction, target):
    prediction = np.asarray(prediction)
    if prediction.shape != target.shape:
        raise ValueError(f"Prediction shape {prediction.shape} does not match target shape {target.shape}")
    if not np.isfinite(prediction).all():
        raise ValueError("Predictions contain non-finite values")

    model_mse = float(np.median(np.mean((target - prediction) ** 2, axis=1)))
    if model_mse <= 0:
        raise ValueError(f"Expected positive median MSE, got {model_mse}")
    baseline_mse = float(np.median(np.mean((target - TRAIN_MEAN) ** 2, axis=1)))
    return baseline_mse / model_mse


# intentional-shortcut: MatGL 0.8.5 creates these tensors on CPU. Keep this
# experiment-local patch until the pinned MatGL stack can be upgraded.
def _gpu_spherical_bessel(self, radius):
    cutoff = torch.as_tensor(self.cutoff, dtype=radius.dtype, device=radius.device)
    roots = matgl_basis.SPHERICAL_BESSEL_ROOTS[: self.max_l, : self.max_n].to(
        radius.device, dtype=radius.dtype
    )
    factor = torch.sqrt(torch.as_tensor(2.0, dtype=radius.dtype, device=radius.device) / cutoff**3)
    radius = radius.clamp(max=cutoff)
    return torch.cat([
        self.funcs[i](radius[:, None] * roots[i][None, :] / cutoff)
        * factor
        / torch.abs(self.funcs[i + 1](roots[i][None, :]))
        for i in range(self.max_l)
    ], dim=1)


def _gpu_combine_basis(sbf, shf, max_n, max_l, use_phi):
    if sbf.size(0) == 0:
        return sbf
    if use_phi:
        repeats = torch.repeat_interleave(2 * torch.arange(max_l, device=sbf.device) + 1, max_n)
        block_sizes = 2 * torch.arange(max_l, device=sbf.device) + 1
    else:
        repeats = torch.ones(max_l * max_n, dtype=torch.long, device=sbf.device)
        block_sizes = [1] * max_l

    columns = torch.arange(shf.size(1), device=shf.device)
    indices, start = [], 0
    for block_size in block_sizes:
        block_size = int(block_size)
        indices.append(torch.tile(columns[start : start + block_size], [max_n]))
        start += block_size

    sbf = torch.repeat_interleave(sbf, repeats, dim=1)
    shf = torch.index_select(shf, 1, torch.cat(indices))
    width = max_n * max_l * (max_l if use_phi else 1)
    return (sbf * shf).reshape(-1, width)


def _gpu_scatter_sum(values, segment_ids, num_segments, dim):
    segment_ids = matgl_math.broadcast(segment_ids.to(values.device), values, dim)
    shape = list(values.size())
    shape[dim] = num_segments if segment_ids.numel() else 0
    return torch.zeros(shape, dtype=values.dtype, device=values.device).scatter_add_(
        dim, segment_ids, values
    )


matgl_basis.SphericalBesselFunction._call_sbf = _gpu_spherical_bessel
matgl_basis.combine_sbf_shf = _gpu_combine_basis
matgl_three_body.combine_sbf_shf = _gpu_combine_basis
matgl_math.scatter_sum = _gpu_scatter_sum
matgl_three_body.scatter_sum = _gpu_scatter_sum


class FEFFDataset(Dataset):
    def __init__(self, split):
        self.rows = []
        for row_id, spectrum in zip(split_ids[split], y_true[split], strict=True):
            material_id, site = row_id.rsplit("_", 1)
            self.rows.append((material_id, int(site), torch.from_numpy(spectrum)))
        self.structures = {}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        material_id, site, spectrum = self.rows[index]
        if material_id not in self.structures:
            path = RAW_ROOT / "FEFF" / "Cu" / material_id / "POSCAR"
            self.structures[material_id] = Structure.from_file(path)
        structure = self.structures[material_id]
        if not 0 <= site < len(structure):
            raise IndexError(f"Absorbing site {site} is outside {material_id} with {len(structure)} sites")
        return structure, site, spectrum


class GraphCollator:
    def __init__(self, encoder):
        self.converter = Structure2Graph(encoder.element_types, encoder.cutoff)

    def __call__(self, batch):
        graphs, sites, spectra = [], [], []
        node_offset = 0
        for structure, site, spectrum in batch:
            converted = self.converter.get_graph(structure)
            graph = converted[0]
            lattice = structure.lattice.matrix if len(converted) == 2 else converted[1]
            lattice = lattice[0] if getattr(lattice, "ndim", 0) == 3 else lattice
            lattice = torch.tensor(np.array(lattice, copy=True), dtype=torch.float32)

            if "pbc_offshift" in graph.edata:
                graph.edata["pbc_offshift"] = graph.edata["pbc_offshift"].float()
            else:
                graph.edata["pbc_offshift"] = graph.edata["pbc_offset"].float() @ lattice

            if "pos" in graph.ndata:
                graph.ndata["pos"] = graph.ndata["pos"].float()
            elif "frac_coords" in graph.ndata:
                graph.ndata["pos"] = graph.ndata["frac_coords"].float() @ lattice
            else:
                graph.ndata["pos"] = torch.as_tensor(structure.cart_coords, dtype=torch.float32)

            graph.edata["bond_vec"], graph.edata["bond_dist"] = compute_pair_vector_and_distance(graph)
            graphs.append(graph)
            sites.append(node_offset + site)
            spectra.append(spectrum)
            node_offset += graph.num_nodes()

        return {
            "graph": dgl.batch(graphs),
            "site": torch.tensor(sites, dtype=torch.long),
            "y": torch.stack(spectra).float(),
        }


In [ ]:
class AttentionReadout(nn.Module):
    def __init__(self):
        super().__init__()
        self.query = nn.Linear(ENCODER_WIDTH, ENCODER_WIDTH, bias=False)
        self.key = nn.Linear(ENCODER_WIDTH, ENCODER_WIDTH, bias=False)
        self.value = nn.Linear(ENCODER_WIDTH, ENCODER_WIDTH, bias=False)

    def forward(self, graph, node, sites):
        source, destination, edge_id = graph.in_edges(sites, form="all")
        batch_row = torch.zeros(graph.num_nodes(), dtype=torch.long, device=node.device)
        batch_row[sites] = torch.arange(len(sites), device=node.device)
        batch_row = batch_row[destination]
        distance = graph.edata["bond_dist"][edge_id]

        logits = (self.query(node[destination]) * self.key(node[source])).sum(-1) / ENCODER_WIDTH**0.5
        logits += torch.log(polynomial_cutoff(distance, CUTOFF).clamp_min(1e-9))
        peak = node.new_full((len(sites),), -torch.inf).scatter_reduce(
            0, batch_row, logits, "amax", include_self=False
        )
        weight = torch.exp(logits - peak[batch_row])
        norm = node.new_zeros(len(sites)).scatter_add(0, batch_row, weight).clamp_min(1e-12)
        context = node.new_zeros(len(sites), ENCODER_WIDTH).index_add(
            0,
            batch_row,
            (weight / norm[batch_row]).unsqueeze(-1) * self.value(node[source]),
        )

        coordination = node.new_zeros(len(sites)).scatter_add(0, batch_row, torch.ones_like(distance))
        coordination3 = node.new_zeros(len(sites)).scatter_add(0, batch_row, (distance <= 3.0).float())
        min_distance = node.new_full((len(sites),), CUTOFF).scatter_reduce(
            0, batch_row, distance, "amin"
        )
        mean_distance = (
            node.new_zeros(len(sites)).scatter_add(0, batch_row, distance)
            / coordination.clamp_min(1)
        )
        geometry = torch.stack(
            [coordination / 10, coordination3 / 10, min_distance / CUTOFF, mean_distance / CUTOFF],
            dim=-1,
        )
        return torch.cat([node[sites], context, geometry], dim=-1)


class XASEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        activation = ActivationFunction["swish"].value()
        degree = 9
        self.element_types = DEFAULT_ELEMENTS
        self.cutoff = CUTOFF
        self.bond_expansion = BondExpansion(3, 3, CUTOFF)
        self.basis_expansion = SphericalBesselWithHarmonics(
            3, 3, CUTOFF, use_smooth=False, use_phi=False
        )
        self.embedding = EmbeddingBlock(
            degree_rbf=degree,
            dim_node_embedding=ENCODER_WIDTH,
            dim_edge_embedding=ENCODER_WIDTH,
            ntypes_node=len(DEFAULT_ELEMENTS),
            activation=activation,
        )
        self.three_body_interactions = nn.ModuleList([
            ThreeBodyInteractions(
                update_network_atom=M3GNetMLP(
                    dims=[ENCODER_WIDTH, degree], activation=nn.Sigmoid(), activate_last=True
                ),
                update_network_bond=GatedMLP(in_feats=degree, dims=[ENCODER_WIDTH], use_bias=False),
            )
            for _ in range(N_BLOCKS)
        ])
        self.graph_layers = nn.ModuleList([
            M3GNetBlock(
                degree=degree,
                activation=activation,
                conv_hiddens=[ENCODER_WIDTH, ENCODER_WIDTH],
                dim_node_feats=ENCODER_WIDTH,
                dim_edge_feats=ENCODER_WIDTH,
                dropout=GNN_DROPOUT,
            )
            for _ in range(N_BLOCKS)
        ])
        self.readout = AttentionReadout()

    def node_features(self, graph):
        graph.edata["rbf"] = self.bond_expansion(graph.edata["bond_dist"])
        line_graph = create_line_graph(graph.to("cpu"), CUTOFF).to(graph.device)
        line_graph.apply_edges(compute_theta_and_phi)
        basis = self.basis_expansion(line_graph)
        cutoff = polynomial_cutoff(graph.edata["bond_dist"], CUTOFF)
        node, edge, state = self.embedding(graph.ndata["node_type"], graph.edata["rbf"], None)
        for three_body, graph_layer in zip(
            self.three_body_interactions, self.graph_layers, strict=True
        ):
            edge = three_body(graph, line_graph, basis, cutoff, node, edge)
            edge, node, state = graph_layer(graph, edge, node, state)
        return node

    def shell_features(self, graph, sites):
        node = self.node_features(graph)
        source, destination, edge_id = graph.in_edges(sites, form="all")
        batch_row = torch.zeros(graph.num_nodes(), dtype=torch.long, device=node.device)
        batch_row[sites] = torch.arange(len(sites), device=node.device)
        batch_row = batch_row[destination]
        distance = graph.edata["bond_dist"][edge_id]

        def weighted_neighbors(mask):
            rows = batch_row[mask]
            weights = 1.0 / distance[mask].clamp_min(1e-6)
            norm = node.new_zeros(len(sites)).scatter_add(0, rows, weights).clamp_min(1e-12)
            return node.new_zeros(len(sites), ENCODER_WIDTH).index_add(
                0,
                rows,
                (weights / norm[rows]).unsqueeze(-1) * node[source[mask]],
            )

        return torch.cat([
            node[sites],
            weighted_neighbors(distance <= 3.0),
            weighted_neighbors((distance > 3.0) & (distance <= CUTOFF)),
        ], dim=-1)

    def forward(self, graph, sites):
        return self.readout(graph, self.node_features(graph), sites)


class LitEncoder(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.encoder = XASEncoder()
        self.head = nn.Sequential(
            nn.Linear(2 * ENCODER_WIDTH + 4, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 141),
            nn.Softplus(),
        )
        self.val_mses = []

    def _shared_step(self, batch, split):
        graph = batch["graph"].to(self.device)
        sites = batch["site"].to(self.device)
        target = batch["y"].to(self.device)
        prediction = self.head(self.encoder(graph, sites) * FEATURE_SCALE)
        loss = ((prediction - target) ** 2).mean()
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite {split} loss")
        self.log(f"{split}_loss", loss, on_epoch=True, prog_bar=True)
        if split == "val":
            self.val_mses.append(((prediction - target) ** 2).mean(dim=1).detach())
        return loss

    def training_step(self, batch, _):
        return self._shared_step(batch, "train")

    def on_validation_epoch_start(self):
        self.val_mses.clear()

    def validation_step(self, batch, _):
        return self._shared_step(batch, "val")

    def on_validation_epoch_end(self):
        if not self.val_mses:
            raise RuntimeError("Validation produced no batches")
        self.log("val_median_mse", torch.cat(self.val_mses).median(), prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=ENCODER_LR, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, ENCODER_EPOCHS, eta_min=1e-6
        )
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"}}


In [ ]:
def prepare_encoder_run(encoder_seed):
    run_dir = OUT_ROOT / f"learned_{ENCODER_WIDTH}d_seed{encoder_seed}"
    checkpoint_dir = run_dir / "encoder_checkpoints"
    feature_file = run_dir / "features.npz"
    done_file = checkpoint_dir / "DONE"
    config_file = run_dir / "config.json"
    encoder_config = {
        "task": TASK,
        "encoder_width": ENCODER_WIDTH,
        "seed": encoder_seed,
        "cutoff": CUTOFF,
        "n_blocks": N_BLOCKS,
        "gnn_dropout": GNN_DROPOUT,
        "epochs": ENCODER_EPOCHS,
        "learning_rate": ENCODER_LR,
        "feature_scale": FEATURE_SCALE,
    }
    if run_dir.exists() and not config_file.exists():
        raise RuntimeError(f"Unversioned encoder run exists; remove or migrate it: {run_dir}")
    if config_file.exists() and json.loads(config_file.read_text()) != encoder_config:
        raise RuntimeError(f"Encoder configuration changed; use a new run directory: {run_dir}")

    if done_file.exists():
        print("encoder cached:", run_dir.name)
    else:
        run_dir.mkdir(parents=True, exist_ok=True)
        config_file.write_text(json.dumps(encoder_config, indent=2) + "\n")
        pl.seed_everything(encoder_seed, workers=True)
        encoder_model = LitEncoder()
        collate = GraphCollator(encoder_model.encoder)
        checkpoint_callback = ModelCheckpoint(
            checkpoint_dir,
            filename="best-{epoch:03d}-{val_median_mse:.5f}",
            monitor="val_median_mse",
            mode="min",
            save_top_k=1,
            save_last=True,
        )
        trainer = pl.Trainer(
            max_epochs=ENCODER_EPOCHS,
            accelerator="auto",
            devices=1,
            callbacks=[checkpoint_callback],
            logger=CSVLogger(str(run_dir), name="encoder_logs", version=0),
            log_every_n_steps=10,
        )
        last_checkpoint = checkpoint_dir / "last.ckpt"
        trainer.fit(
            encoder_model,
            DataLoader(
                FEFFDataset("train"), BATCH_SIZE, shuffle=True,
                collate_fn=collate, num_workers=NUM_WORKERS,
            ),
            DataLoader(
                FEFFDataset("val"), BATCH_SIZE, shuffle=False,
                collate_fn=collate, num_workers=NUM_WORKERS,
            ),
            ckpt_path=str(last_checkpoint) if last_checkpoint.exists() else None,
        )
        if (
            not Path(checkpoint_callback.best_model_path).is_file()
            or checkpoint_callback.best_model_score is None
            or not torch.isfinite(checkpoint_callback.best_model_score)
        ):
            raise RuntimeError(f"Encoder did not produce a finite validation-best checkpoint: {run_dir}")
        done_file.write_text("ok\n")
        del encoder_model, trainer
        torch.cuda.empty_cache()

    best_checkpoints = list(checkpoint_dir.glob("best-*.ckpt"))
    if not done_file.exists() or len(best_checkpoints) != 1:
        raise RuntimeError(
            f"Expected one completed validation-best encoder in {checkpoint_dir}; "
            f"found {len(best_checkpoints)}"
        )
    best_checkpoint = best_checkpoints[0]
    checkpoint_stamp = np.int64(best_checkpoint.stat().st_mtime_ns)

    if feature_file.exists():
        with np.load(feature_file, allow_pickle=False) as cached:
            valid = int(cached["checkpoint_stamp"]) == int(checkpoint_stamp)
            for split in SPLITS:
                key = f"{FEATURE_KEY}_{split}"
                expected_shape = (len(split_ids[split]), FEATURE_DIM)
                valid &= key in cached and cached[key].shape == expected_shape
                valid &= key in cached and np.isfinite(cached[key]).all()
                valid &= f"ids_{split}" in cached and np.array_equal(
                    cached[f"ids_{split}"], split_ids[split]
                )
        if not valid:
            raise RuntimeError(f"Stale or invalid feature cache: {feature_file}")
        print("features cached:", feature_file)
    else:
        state = torch.load(best_checkpoint, map_location="cpu")["state_dict"]
        encoder_model = LitEncoder()
        encoder_model.load_state_dict(state)
        encoder = encoder_model.encoder.to(DEVICE).eval()
        collate = GraphCollator(encoder)
        arrays = {f"ids_{split}": np.asarray(split_ids[split]) for split in SPLITS}
        with torch.no_grad():
            for split in SPLITS:
                chunks = []
                loader = DataLoader(
                    FEFFDataset(split), BATCH_SIZE, shuffle=False,
                    collate_fn=collate, num_workers=NUM_WORKERS,
                )
                for batch in loader:
                    features = encoder.shell_features(
                        batch["graph"].to(DEVICE), batch["site"].to(DEVICE)
                    )
                    chunks.append((features * FEATURE_SCALE).cpu().numpy())
                features = np.concatenate(chunks)
                expected_shape = (len(split_ids[split]), FEATURE_DIM)
                if features.shape != expected_shape or not np.isfinite(features).all():
                    raise RuntimeError(
                        f"{split}: expected finite features with shape {expected_shape}, "
                        f"got {features.shape}"
                    )
                arrays[f"{FEATURE_KEY}_{split}"] = features
        np.savez_compressed(feature_file, checkpoint_stamp=checkpoint_stamp, **arrays)
        del encoder_model, encoder
        torch.cuda.empty_cache()
        print("wrote:", feature_file)

    return {
        "run_dir": run_dir,
        "feature_file": feature_file,
        "checkpoint_stamp": checkpoint_stamp,
    }


encoder_runs = {seed: prepare_encoder_run(seed) for seed in ENCODER_SEEDS}


In [ ]:
def load_encoder_features(run_info):
    feature_file = run_info["feature_file"]
    checkpoint_stamp = run_info["checkpoint_stamp"]
    with np.load(feature_file, allow_pickle=False) as saved:
        if int(saved["checkpoint_stamp"]) != int(checkpoint_stamp):
            raise RuntimeError(f"Encoder provenance mismatch: {feature_file}")
        result = {}
        for split in SPLITS:
            if not np.array_equal(saved[f"ids_{split}"], split_ids[split]):
                raise RuntimeError(f"{split}: ID order mismatch in {feature_file}")
            features = saved[f"{FEATURE_KEY}_{split}"].copy()
            expected_shape = (len(split_ids[split]), FEATURE_DIM)
            if features.shape != expected_shape or not np.isfinite(features).all():
                raise RuntimeError(f"{split}: expected finite features with shape {expected_shape}")
            result[split] = features
    return result


def train_heads(encoder_seed, run_info):
    features_by_split = load_encoder_features(run_info)
    feature_id = (
        f"{run_info['run_dir'].name}:{int(run_info['checkpoint_stamp'])}:{FEATURE_KEY}"
    )
    head_root = run_info["run_dir"] / FEATURE_KEY / "heads"
    ml_splits = MLSplits(**{
        split: MLData(X=features_by_split[split], y=y_true[split])
        for split in SPLITS
    })

    for head_config in HEAD_CONFIGS:
        run_name = head_run_name(head_config)
        head_dir = head_root / run_name
        prediction_file = head_dir / "predictions.npz"
        config = {
            **head_config,
            "encoder_seed": encoder_seed,
            "feature_id": feature_id,
            "input_dim": FEATURE_DIM,
            "hidden_dims": HEAD_DIMS,
            "output_dim": 141,
            "epochs": HEAD_EPOCHS,
        }
        config_json = json.dumps(config, sort_keys=True)

        if prediction_file.exists():
            with np.load(prediction_file, allow_pickle=False) as saved:
                valid = str(saved["config"]) == config_json
                for split in ("val", "test"):
                    valid &= saved[split].shape == y_true[split].shape
                    valid &= np.isfinite(saved[split]).all()
            if not valid:
                raise RuntimeError(f"Invalid cached predictions: {prediction_file}")
            print("head cached:", encoder_seed, run_name)
            continue
        if head_dir.exists():
            raise RuntimeError(f"Incomplete head run; remove it before retrying: {head_dir}")

        head_dir.mkdir(parents=True)
        pl.seed_everything(head_config["seed"], workers=True)
        XASBlock.DROPOUT = head_config["dropout"]
        regressor = XASBlockRegressor(
            directory=str(head_dir),
            input_dim=FEATURE_DIM,
            hidden_dims=HEAD_DIMS,
            output_dim=141,
            initial_lr=head_config["lr"],
            batch_size=BATCH_SIZE,
            max_epochs=HEAD_EPOCHS,
            use_early_stopping=False,
            use_lr_finder=False,
            monitor_metric="val_median_mse",
            shuffle=True,
            lr_scheduler="cosine",
            cosine_t_max=HEAD_EPOCHS,
            cosine_eta_min=1e-6,
            overwrite_save_dir=False,
        )
        regressor.fit(ml_splits).load("best")
        model = regressor.model.model.to(DEVICE).eval()
        predictions = {"config": np.asarray(config_json)}
        with torch.no_grad():
            for split in ("val", "test"):
                features = features_by_split[split]
                predictions[split] = np.concatenate([
                    model(torch.as_tensor(
                        features[start : start + 512], dtype=torch.float32, device=DEVICE
                    )).cpu().numpy()
                    for start in range(0, len(features), 512)
                ])
                if (
                    predictions[split].shape != y_true[split].shape
                    or not np.isfinite(predictions[split]).all()
                ):
                    raise RuntimeError(
                        f"Invalid predictions: encoder {encoder_seed} {run_name} {split}"
                    )
        np.savez_compressed(prediction_file, **predictions)
        print("encoder", encoder_seed, run_name, {
            split: round(eta_score(predictions[split], y_true[split]), 3)
            for split in ("val", "test")
        })
        del model, regressor
        torch.cuda.empty_cache()

    return {"feature_id": feature_id, "head_root": head_root}


experiment_data = {
    seed: train_heads(seed, encoder_runs[seed])
    for seed in ENCODER_SEEDS
}


In [ ]:
member_rows = []
prediction_records = []

for encoder_seed, data in experiment_data.items():
    for head_config in HEAD_CONFIGS:
        run_name = head_run_name(head_config)
        prediction_file = data["head_root"] / run_name / "predictions.npz"
        if not prediction_file.exists():
            raise FileNotFoundError(f"Missing head predictions: {prediction_file}")
        with np.load(prediction_file, allow_pickle=False) as saved:
            config = json.loads(str(saved["config"]))
            if config["feature_id"] != data["feature_id"] or config["encoder_seed"] != encoder_seed:
                raise RuntimeError(f"Feature provenance mismatch: {prediction_file}")
            val_prediction = saved["val"].copy()
            test_prediction = saved["test"].copy()
        member_rows.append({
            "encoder_seed": encoder_seed,
            "head": run_name,
            **head_config,
            "val_eta": eta_score(val_prediction, y_true["val"]),
            "test_eta": eta_score(test_prediction, y_true["test"]),
        })
        prediction_records.append({
            "encoder_seed": encoder_seed,
            "val": val_prediction,
            "test": test_prediction,
        })

members = pd.DataFrame(member_rows)
expected_members = len(ENCODER_SEEDS) * len(HEAD_CONFIGS)
if len(members) != expected_members:
    raise RuntimeError(f"Expected {expected_members} members, found {len(members)}")

combo_predictions = {}
combo_rows = []
for encoder_seed in ENCODER_SEEDS:
    indices = members.index[members["encoder_seed"] == encoder_seed].to_list()
    val_prediction = np.stack([prediction_records[i]["val"] for i in indices]).mean(axis=0)
    test_prediction = np.stack([prediction_records[i]["test"] for i in indices]).mean(axis=0)
    name = f"encoder_seed{encoder_seed}_10_head_ensemble"
    combo_predictions[name] = {"val": val_prediction, "test": test_prediction}
    combo_rows.append({
        "combo": name,
        "n_members": len(indices),
        "val_eta": eta_score(val_prediction, y_true["val"]),
        "test_eta": eta_score(test_prediction, y_true["test"]),
    })

all_val_prediction = np.stack([record["val"] for record in prediction_records]).mean(axis=0)
all_test_prediction = np.stack([record["test"] for record in prediction_records]).mean(axis=0)
combo_predictions["all_20_heads"] = {"val": all_val_prediction, "test": all_test_prediction}
combo_rows.append({
    "combo": "all_20_heads",
    "n_members": len(prediction_records),
    "val_eta": eta_score(all_val_prediction, y_true["val"]),
    "test_eta": eta_score(all_test_prediction, y_true["test"]),
})

best_member_index = int(members["val_eta"].idxmax())
combo_predictions["best_validation_head"] = {
    "val": prediction_records[best_member_index]["val"],
    "test": prediction_records[best_member_index]["test"],
}
combo_rows.append({
    "combo": "best_validation_head",
    "n_members": 1,
    "val_eta": members.loc[best_member_index, "val_eta"],
    "test_eta": members.loc[best_member_index, "test_eta"],
})

selection_summary = pd.DataFrame(combo_rows).sort_values(
    "val_eta", ascending=False, ignore_index=True
)
summary = pd.concat([
    pd.DataFrame([
        {"combo": "paper ExpertXAS", "n_members": 1, "val_eta": np.nan, "test_eta": PAPER_EXPERT_ETA},
        {
            "combo": "old v1 20-member ensemble",
            "n_members": 20,
            "val_eta": OLD_V1_VAL_ETA,
            "test_eta": OLD_V1_TEST_ETA,
        },
    ]),
    selection_summary,
], ignore_index=True)

members.to_csv(OUT_ROOT / "members.csv", index=False)
summary.to_csv(OUT_ROOT / "summary.csv", index=False)
selection_summary.to_csv(OUT_ROOT / "combo_summary.csv", index=False)
display(members.sort_values(["encoder_seed", "val_eta"], ascending=[True, False]).round(3))
display(summary.round(3))

selected_name = selection_summary.loc[0, "combo"]
selected_test_prediction = combo_predictions[selected_name]["test"]
baseline_mse = np.mean((y_true["test"] - TRAIN_MEAN) ** 2, axis=1)
selected_mse = np.mean((y_true["test"] - selected_test_prediction) ** 2, axis=1)
print("validation-selected result:", selected_name)


## Visual diagnostics

The result is fixed by validation eta before these test diagnostics are produced. Do not use the plots to revise model or ensemble selection.


In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 4.8), dpi=150)
ax.scatter(
    members["val_eta"], members["test_eta"],
    s=45, alpha=0.8, color="#2a78d6", edgecolors="white",
)
low = members[["val_eta", "test_eta"]].min().min() - 0.2
high = members[["val_eta", "test_eta"]].max().max() + 0.2
ax.plot([low, high], [low, high], linestyle=":", color="gray")
ax.set(
    xlabel="expert validation eta",
    ylabel="expert test eta",
    title="Expert validation/test variation across two encoders",
)
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_ROOT / "member_val_test_scatter.png", dpi=220)
plt.show()

if (baseline_mse <= 0).any() or (selected_mse <= 0).any():
    raise ValueError("Log-error plot requires positive per-spectrum MSE values")
improved = float(np.mean(selected_mse < baseline_mse))
fig, ax = plt.subplots(figsize=(5.8, 5.2), dpi=150)
ax.scatter(baseline_mse, selected_mse, s=14, alpha=0.5, color="#2a78d6", edgecolors="none")
low = min(baseline_mse.min(), selected_mse.min())
high = max(baseline_mse.max(), selected_mse.max())
ax.plot([low, high], [low, high], color="gray", linestyle=":", linewidth=1.5)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(
    xlabel="mean-spectrum baseline MSE",
    ylabel=f"{selected_name} MSE",
    title=f"Validation-selected result vs baseline ({improved:.1%} improved)",
)
ax.grid(alpha=0.25, which="both")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT_ROOT / "validation_selected_combo_vs_baseline.png", dpi=220)
plt.show()
